# 🛡️ Bug Bounty Copilot - Autonomous Training Pipeline (Kaggle T4 GPU)

This notebook trains an advanced offensive security reasoning model using **Unsloth (QLoRA 4-bit)** on **Qwen2.5-7B-Instruct**.
- **Dataset**: Curated PentesterLand Bug Bounty Chain-of-Thought (CoT) pairs.
- **Hardware**: Kaggle Free T4 GPU (Background Execution Supported).
- **Output**: LoRA adapter saved to `/kaggle/working/bugbounty_copilot_lora`.

In [ ]:
# 1. Install Unsloth & Core Dependencies (Optimized for T4 GPU)
!pip install -q --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q unsloth_zoo
!pip install -q --no-deps trl peft accelerate bitsandbytes
!pip install -q datasets transformers triton xformers


In [ ]:
# 2. Check for Previous Accumulated LoRA Weights (Mount & Resume)
from unsloth import FastLanguageModel
import torch, os, glob, urllib.request, json

max_seq_length = 4096
dtype = None # Auto detection (Float16 for T4)
load_in_4bit = True

# Injected by orchestrator before push (signed direct download URLs)
LATEST_LORA_URLS = {}

# Priority 1: Check mounted kernel sources in /kaggle/input/
possible_loras = glob.glob("/kaggle/input/**/bugbounty_copilot_lora/adapter_model.safetensors", recursive=True)
if not possible_loras:
    possible_loras = glob.glob("/kaggle/input/**/adapter_model.safetensors", recursive=True)

# Priority 2: Direct signed URL download from partner node
if not possible_loras and LATEST_LORA_URLS:
    print(f"[*] Downloading {len(LATEST_LORA_URLS)} accumulated LoRA files from previous run...")
    lora_dir = "/kaggle/working/latest_accumulated_lora"
    os.makedirs(lora_dir, exist_ok=True)
    for fname, url in LATEST_LORA_URLS.items():
        local_f = os.path.join(lora_dir, os.path.basename(fname))
        try:
            urllib.request.urlretrieve(url, local_f)
            print(f"    -> Downloaded: {os.path.basename(fname)}")
        except Exception as e:
            print(f"    -> Warning downloading {fname}: {e}")
    if os.path.exists(os.path.join(lora_dir, "adapter_model.safetensors")):
        possible_loras = [os.path.join(lora_dir, "adapter_model.safetensors")]

# Priority 3: Fallback via CLI
if not possible_loras:
    print("[*] Searching for partner node weights via Kaggle direct output pull...")
    os.system("kaggle kernels output mmossad2324/bugbounty-copilot-trainer -p /kaggle/working/prev_acc3 >/dev/null 2>&1 || true")
    os.system("kaggle kernels output mmossad2124/bugbounty-copilot-trainer -p /kaggle/working/prev_acc1 >/dev/null 2>&1 || true")
    candidates = glob.glob("/kaggle/working/prev_acc*/**/adapter_model.safetensors", recursive=True)
    if candidates:
        candidates.sort(key=lambda x: os.path.getmtime(x), reverse=True)
        possible_loras = [candidates[0]]

loaded_from_previous = False
if possible_loras:
    prev_adapter_dir = os.path.dirname(possible_loras[0])
    print(f"[✓] FOUND PREVIOUS DAY ACCUMULATED WEIGHTS at: {prev_adapter_dir}")
    print("[*] Resuming and stacking incremental training on top of previous day progress...")
    try:
        model, tokenizer = FastLanguageModel.from_pretrained(
            model_name = prev_adapter_dir,
            max_seq_length = max_seq_length,
            dtype = dtype,
            load_in_4bit = load_in_4bit,
        )
        FastLanguageModel.for_training(model)
        loaded_from_previous = True
        print("[✓] Successfully loaded previous day accumulated weights! Knowledge is compounding.")
    except Exception as e:
        print(f"[!] Could not load previous weights directly ({e}). Falling back to base model.")

if not loaded_from_previous:
    model_name = "Qwen/Qwen2.5-7B-Instruct"
    print(f"[*] Initializing fresh base model: {model_name} in 4-bit...")
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = model_name,
        max_seq_length = max_seq_length,
        dtype = dtype,
        load_in_4bit = load_in_4bit,
    )
    model = FastLanguageModel.get_peft_model(
        model,
        r = 16,
        target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                          "gate_proj", "up_proj", "down_proj"],
        lora_alpha = 16,
        lora_dropout = 0,
        bias = "none",
        use_gradient_checkpointing = "unsloth",
        random_state = 3407,
        use_rslora = False,
        loftq_config = None,
    )
    print("[✓] Attached fresh QLoRA adapters to base model!")


In [ ]:
# 3. Verify Active LoRA Training Status
FastLanguageModel.for_training(model)
print('[✓] Model & LoRA Adapters verified in active training mode!')


In [ ]:
# 4. Load & Format Training Dataset (ChatML Format with Dynamic Relay Specialization)
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template = "chatml", # Qwen2.5 standard
)

def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = [tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = False) for convo in convos]
    return { "text" : texts }

# Dynamic Relay Specialization:
# Account 1 (mmossad2124) -> Stage 1: Web, Cloud, AI & Bug Bounty (20,895 samples)
# Account 3 (mmossad2324) -> Stage 2: Kernel, Memory Safety, APTs & Evasion (6,022 samples)
ACTIVE_NODE_USER = "mmossad2124" # Daemon updates this to mmossad2124 or mmossad2324

# Auto-detect node identity from mounted input if available
if any("mmossad2124" in p for p in glob.glob("/kaggle/input/*")):
    ACTIVE_NODE_USER = "mmossad2324"
elif any("mmossad2324" in p for p in glob.glob("/kaggle/input/*")):
    ACTIVE_NODE_USER = "mmossad2124"

if ACTIVE_NODE_USER == "mmossad2324":
    dataset_file = "stage2_kernel_apt_arsenal.jsonl"
    print("[*] Node Specialization: Account 3 (Stage 2: Kernel, Memory, APTs & Evasion Arsenal - 6,022 samples)")
else:
    dataset_file = "stage1_web_cloud_arsenal.jsonl"
    print("[*] Node Specialization: Account 1 (Stage 1: Web, Cloud, AI & Bug Bounty Arsenal - 20,895 samples)")

dataset_url = f"https://raw.githubusercontent.com/mmossad2124-blip/BugBounty-Copilot/main/data_pipeline/processed_datasets/{dataset_file}"
print(f"[*] Loading specialized dataset from: {dataset_url}")
dataset = load_dataset("json", data_files=dataset_url, split="train")
dataset = dataset.map(formatting_prompts_func, batched = True)
print(f"[✓] Successfully loaded {len(dataset)} elite offensive reasoning samples!")


In [ ]:
# 5. Configure SFTTrainer (Supervised Fine-Tuning)
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 10,
        max_steps = 300,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "/kaggle/working/checkpoints",
        save_steps = 100,
    ),
)
print("[✓] Trainer configured and ready!")

In [ ]:
# 6. Train the Model!
print("[*] Starting training loop on GPU...")
trainer_stats = trainer.train()
print("[✓] Training completed successfully!")

In [ ]:
# 7. Test Inference (Sample Vulnerability Reasoning)
FastLanguageModel.for_inference(model)

messages = [
    {"role": "system", "content": "You are an Elite Offensive Security & Bug Bounty AI Copilot."},
    {"role": "user", "content": "Target scope: api.target.com/v1/user/profile. How would you methodically test for IDOR and privilege escalation?"}
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True,
    return_tensors = "pt",
).to("cuda")

outputs = model.generate(input_ids = inputs, max_new_tokens = 512, use_cache = True)
print(tokenizer.decode(outputs[0], skip_special_tokens = True))

In [ ]:
# 8. Save LoRA Adapters to Kaggle Output
output_dir = "/kaggle/working/bugbounty_copilot_lora"
print(f"[*] Saving trained LoRA adapter to {output_dir}...")
model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)
print("[✓] LoRA adapters permanently saved to output! Available in Kaggle output tab.")